# Exploratory Data Analysis

## Imports

In [2]:
from pathlib import Path

import altair as alt
import pandas as pd
from IPython.display import display

NAVY = "#1f3864"

## Load Clean Data

In [3]:
cwd = Path.cwd()
root = next(p for p in [cwd, *cwd.parents] if (p / "data" / "raw" / "survey_data.csv").exists())

df_clean = pd.read_csv(root / "data" / "clean" / "clean_data.csv")
codelist = pd.read_csv(root / "data" / "raw" / "codelist.csv")

display(df_clean.head())

,respondent,sex,location,q1,q2,q3,q4,q5
0,1,M,MT,4,3,1,Climate Change,5
1,2,F,CO,2,1,1,Environmental Justice,2
2,3,M,CO,3,3,2,Clean Air & Water,4
3,4,M,MT,3,1,1,Clean Air & Water,2
4,5,M,MT,2,2,1,Land & Wildlife,1


## Exploratory Summary

Includes row counts, unique values, and a 5-number summary for each variable (if applicable)

In [4]:
display(df_clean.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
respondent,100.0,NaN,NaN,NaN,50.5,29.011492,1.0,25.75,50.5,75.25,100.0
sex,100,2,F,63,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location,100,2,CO,64,NaN,NaN,NaN,NaN,NaN,NaN,NaN
q1,100.0,NaN,NaN,NaN,2.82,1.290055,1.0,2.0,3.0,4.0,5.0
q2,100.0,NaN,NaN,NaN,2.62,1.324059,1.0,1.0,2.0,4.0,5.0
q3,100.0,NaN,NaN,NaN,1.87,1.021536,1.0,1.0,2.0,3.0,5.0
q4,100,5,Land & Wildlife,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
q5,100.0,NaN,NaN,NaN,3.15,1.402559,1.0,2.0,3.0,4.0,5.0


## Overall Response Distributions

These bar charts show the overall response distributions for each of the 5 questions (with their original answers mapped to the numeric codes)

In [ ]:
codelist.columns = [str(c).strip() for c in codelist.columns]
for q in ["q1", "q2", "q3", "q5"]:
    code_row = codelist.loc[codelist.iloc[:, 0].astype(str).str.strip().str.upper() == q.upper()].iloc[0]
    label_map = {int(code): str(code_row[str(code)]).strip() for code in range(1, 6)}
    plot_df = df_clean[q].value_counts(dropna=False).rename_axis("code").reset_index(name="count")
    plot_df["response"] = plot_df["code"].map(label_map).fillna("Missing")
    order = [label_map[i] for i in range(1, 6)]
    if "Missing" in plot_df["response"].values:
        order.append("Missing")
    chart = alt.Chart(plot_df).mark_bar().encode(x=alt.X("response:N", sort=order, title=None), y=alt.Y("count:Q", title="Respondents"), color=alt.value(NAVY), tooltip=["response", "count"]).properties(title=f"Distribution of {q.upper()}", width=650, height=300)
    display(chart)

    ### Q1: We see a pretty even IRA awareness distribution of responses across the 5-point scale
    ### Q2: Most respondents were either very aware or somewhat aware of Green Machine
    ### Q3: Most respondents were aware of the specific organizations
    ### Q5: Pretty even distribution of responses, although many were unsure if they would donate to an environmental org next year. 

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

## Likelihood to Donate by Sex

This chart breaks down the overall 'likeliness' of being willing to donate to an environmental organization by sex. Likeliness was defined by answering either 'extremely likely' or 'very likely' to Q5. 

In [ ]:
sex_map = {"F": "Female", "M": "Male"}
plot_df = df_clean[["sex", "q5"]].dropna().copy()
plot_df["Sex"] = plot_df["sex"].map(sex_map).fillna(plot_df["sex"])
plot_df["likely_to_donate"] = plot_df["q5"].le(2)
summary = plot_df.groupby("Sex", as_index=False).agg(likely_percentage=("likely_to_donate", lambda s: s.mean() * 100), n=("q5", "size"))
summary["label"] = summary["likely_percentage"].round(1).astype(str) + "%"
chart = alt.Chart(summary).mark_bar().encode(x=alt.X("Sex:N", title=None), y=alt.Y("likely_percentage:Q", title="Likely to donate (%)", scale=alt.Scale(domain=[0, 100])), color=alt.Color("Sex:N", legend=None), tooltip=["Sex", alt.Tooltip("likely_percentage:Q", format=".1f"), "n"]).properties(width=650, height=300)
labels = chart.mark_text(dy=-8).encode(text="label:N")
display(chart + labels)

alt.LayerChart(...)

The plot above shows that overall, females were more likely to be willing to donate to an environmental organization next year compared to men. This is NOT a causal claim, but we should investigate what is driving this difference.